This notebook cleans up the Puerto Rico monthly generation fuel data.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
def load_generation_data(path):
    """Load the cleaned Puerto Rico generator operations data from disk.
    
    Args:
        path: Path to raw data file on disk.
    """
    return pd.read_parquet(path)

def write_generation_data(df, path):
    df.to_parquet(path)
    return


def map_code_to_strings(df, mapped_col, code_dictionary):
    """Convert a column of codes into strings defined by a dictionary.

    This code takes a dataframe with a column of codes and returns a dataframe with the same column
    mapped to strings to prevent users from needing to consult a look-up table. The relationship between columns and strings is defined by a dictionary.

    Args:
        df: A Pandas DataFrame.
        mapped_col: The name of the column to be mapped.
        code_dictionary: A dictionary containing code-string pairs.
    """
    assert all([code in code_dictionary for code in df[mapped_col].unique()]) # Check all codes present
    df['code_name'] = df[mapped_col].replace(code_dictionary)
    df = df.drop(columns=mapped_col).rename(columns={'code_name':mapped_col})
    return df

def handle_thousands_fuel_units(df, thousands_unit: str):
    """Where the fuel unit column contains a small number of thousands, multiply by 1000 and update the unit label.

    This expects the thousands_unit to be in a column called "fuel_unit", and the units column to live in a
    column called "fuel_consumed_for_electricity_units".

    Args:
        df: A Pandas DataFrame.
        unit_column: The column containing unit names.
        thousands_unit: The name of the unit to be normalized.
    """
    df.loc[df.fuel_unit == thousands_unit,"fuel_consumed_for_electricity_units"] = df.loc[df.fuel_unit == thousands_unit, "fuel_consumed_for_electricity_units"]*1000
    df.loc[df.fuel_unit == thousands_unit, "fuel_unit"] = df.loc[df.fuel_unit == thousands_unit].fuel_unit.str.replace("thousand ", "")
    return df

In [ ]:
raw_pr_gen_fuel = load_generation_data(path="data/pr_gen_fuel_monthly.parquet")

In [ ]:
# Standardize NAs
pr_gen_fuel = raw_pr_gen_fuel.replace(to_replace = ".", value = pd.NA)
pr_gen_fuel = pr_gen_fuel.replace(to_replace = "null", value = pd.NA)

# convert codes to strings
ENERGY_SOURCE_DICT = {'WND':'wind', 'NG':'natural_gas', 'SUN':'solar',
                    'BIT':'bituminous_coal', 'MWH':"electricity_for_energy_storage",
                    'DFO':'distillate_fuel_oil', 'RFO':'residual_fuel_oil', 'WAT':'hydro'}

PRIME_MOVER_CODE_DICT = {
    'WT':'onshore_wind', 'CA':'cc_steam', 'CT':'cc_combustion_turbine',
    'PV':'photovoltaic', 'ST':'steam_turbine', 'BA':'battery',
    'IC': 'internal_combustion', 'GT': 'gas_turbine', 'HY':'hydraulic_turbine' 
}

pr_gen_fuel = map_code_to_strings(df = pr_gen_fuel, mapped_col = "energy_source_code", code_dictionary = ENERGY_SOURCE_DICT)
pr_gen_fuel = map_code_to_strings(df = pr_gen_fuel, mapped_col = "prime_mover_code", code_dictionary = PRIME_MOVER_CODE_DICT)

pr_gen_fuel = handle_thousands_fuel_units(pr_gen_fuel, thousands_unit = "thousand_short_tons")
pr_gen_fuel = handle_thousands_fuel_units(pr_gen_fuel, thousands_unit = "thousand barrels")

# drop after 2025-03-01 (for now) as these values should not exist
pr_gen_fuel = pr_gen_fuel.loc[pr_gen_fuel.date < pd.Timestamp("2025-03-01")]

In [ ]:
### save cleaned file
write_generation_data(pr_gen_fuel, "data/pr_gen_fuel_monthly_clean.parquet")